# M2 — Fine-tuning do BERTimbau para NER (4 entidades)

Treina o BERTimbau (`neuralmind/bert-base-portuguese-cased`) como classificador de tokens (NER) para reconhecer `CPF`, `RG`, `NOME` e `ENDERECO`, usando o gabarito alinhado (`04.gabarito_alinhado/`) como dataset — juntando documentos **reais** (`04.gabarito_alinhado/gemini-2.5-flash/`, via notebook 03) e **sintéticos** (`04.gabarito_alinhado/sintetico/`, via notebook 00) no mesmo treino.

**CIN foi fundida com CPF (decisão 2026-09-09, ver PRD.md seção 8):** o número da Carteira de Identidade Nacional é literalmente o mesmo número do CPF do titular (Decreto 10.977/2022) — não é um formato numérico próprio para o modelo aprender separadamente. Manter CIN como uma 5ª classe resultava numa categoria sempre vazia (0 exemplos reais até agora), sem ganho prático: qualquer menção futura a "CIN" já é capturada pelo label `CPF`, já que o padrão de dígitos e a validação por checksum são idênticos.

**Metodologia dos dados:**
1. Cada ENTIDADE do gabarito alinhado (real, com bbox do notebook 03, ou sintética, com bbox exato do notebook 00) é convertida em tags BIO (`B-TIPO`/`I-TIPO`/`O`) sobre os tokens da extração local (`03.extracao_ocr_local/`) — mapeando bbox → índice de token por contenção do centro do token na caixa da entidade (a mesma lógica geométrica inversa dos notebooks anteriores).
2. Só entram no treino documentos com **pelo menos `LIMIAR_ALINHAMENTO_MINIMO_PCT`% das entidades alinhadas** (`match_type != "unmatched"`) — documentos sintéticos sempre passam (100% exato por construção); documentos reais com alinhamento muito ruim (notebook 03) são descartados, para não introduzir bbox faltando/errada direto no treino.
3. Cada **página** vira um exemplo de treino (documentos completos excedem o limite de contexto do BERT).
4. Split por **documento** (não por página, para não vazar o mesmo documento entre treino e teste): `FRACAO_TREINO` (80%) dos documentos vão para treino, o restante (20%) para teste.

**Pré-requisitos:** `pip install transformers torch seqeval` (no venv GERAL do projeto, `.venv` — nunca no `.venv_ocr`, ver restrição em `requirements_ocr.txt`).

In [2]:
import json
import random
from pathlib import Path

import numpy as np
import torch
from seqeval.metrics import f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Rodando em CPU — fine-tuning de transformer vai ser bem mais lento.")

d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA disponível: True
GPU: NVIDIA GeForce RTX 5060 Ti


## Configuração

In [3]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()

import sys
sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import comprimentos_subtokens, dividir_em_janelas_por_orcamento

MODELO_GABARITO = "gemini-2.5-flash"
INCLUIR_SINTETICOS = False  # False = treina só com documentos reais (experimento de comparação,
                            # ver anotacoes.txt 2026-09-11: F1 real=0.721 vs. F1 sintético=1.000
                            # no treino misto -- este flag isola o efeito real do sintético)

PASTA_EXTRACAO = RAIZ / "data" / "documento_social" / "03.extracao_ocr_local"
# Gabarito dos documentos reais (via Gemini + alinhamento, notebook 03) e dos sintéticos
# (gerado direto, notebook 00) ficam em subpastas irmãs — juntamos as duas no treino, a não
# ser que INCLUIR_SINTETICOS=False.
PASTA_GABARITO_REAIS = RAIZ / "data" / "documento_social" / "04.gabarito_alinhado" / MODELO_GABARITO
PASTA_GABARITO_SINTETICOS = RAIZ / "data" / "documento_social" / "04.gabarito_alinhado" / "sintetico"
PASTA_MODELO_SAIDA = RAIZ / "models" / "bertimbau-pii-v1"
PASTA_MODELO_SAIDA.mkdir(parents=True, exist_ok=True)

CHECKPOINT_BASE = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_BASE)  # carregado aqui (não só na célula de
                                                              # tokenização) porque a divisão em
                                                              # janelas da preparação do dataset já
                                                              # precisa medir subtokens por palavra

TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]  # CIN fundida com CPF (decisão 2026-09-09, ver PRD.md secao 8)
LABEL_LIST = ["O"] + [f"{prefixo}-{tipo}" for tipo in TIPOS for prefixo in ("B", "I")]
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

LIMIAR_ALINHAMENTO_MINIMO_PCT = 80  # documento só entra no treino se >= X% das suas entidades alinharam (notebook 03)
                                     # documentos sintéticos sempre passam (match_type='exact' em tudo)
FRACAO_TREINO = 0.8                 # split por documento: 80% treino / 20% teste
MAX_LENGTH = 512
SOBREPOSICAO_PALAVRAS_JANELA = 64   # páginas que passam do orçamento de subtokens viram várias
                                     # janelas (ver construir_exemplos_documento) -- mesmo valor
                                     # usado na inferência (notebooks 10/11/12), achado 2026-09-15:
                                     # sem isso, ~28% das entidades do gabarito real nunca geravam
                                     # gradiente (caíam depois do corte de MAX_LENGTH silenciosamente)

IOU_MINIMO_MATCH = 0.30  # mesmo limiar dos notebooks 04/05, para comparabilidade

print(f"Raiz do projeto: {RAIZ}")
print(f"Extração:               {PASTA_EXTRACAO}")
print(f"Gabarito (reais):       {PASTA_GABARITO_REAIS}")
print(f"Gabarito (sintéticos):  {PASTA_GABARITO_SINTETICOS} (INCLUIR_SINTETICOS={INCLUIR_SINTETICOS})")
print(f"Modelo:                 {PASTA_MODELO_SAIDA}")
print(f"Labels ({len(LABEL_LIST)}): {LABEL_LIST}")

Raiz do projeto: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3
Extração:               d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\03.extracao_ocr_local
Gabarito (reais):       d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\04.gabarito_alinhado\gemini-2.5-flash
Gabarito (sintéticos):  d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\04.gabarito_alinhado\sintetico (INCLUIR_SINTETICOS=False)
Modelo:                 d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1
Labels (9): ['O', 'B-CPF', 'I-CPF', 'B-RG', 'I-RG', 'B-NOME', 'I-NOME', 'B-ENDERECO', 'I-ENDERECO']


## Preparação do dataset: bbox → tags BIO por página

In [4]:
def bbox_contem_centro(caixa: list, bbox_token: list) -> bool:
    """Verifica se o centro do token cai dentro da caixa (com folga de 1pt para arredondamento)."""
    x0, y0, x1, y1 = caixa
    cx = (bbox_token[0] + bbox_token[2]) / 2
    cy = (bbox_token[1] + bbox_token[3]) / 2
    return (x0 - 1) <= cx <= (x1 + 1) and (y0 - 1) <= cy <= (y1 + 1)


def indices_tokens_da_entidade(tokens: list, entidade: dict) -> list:
    caixas = entidade.get("bbox") or []
    indices = []
    for i, tok in enumerate(tokens):
        for caixa in caixas:
            if bbox_contem_centro(caixa, tok["bbox"]):
                indices.append(i)
                break
    return sorted(set(indices))


def construir_exemplos_documento(extracao: dict, gabarito: dict) -> list:
    entidades_por_pagina = {}
    for e in gabarito["entities"]:
        if e.get("match_type") == "unmatched":
            continue
        entidades_por_pagina.setdefault(e["page"], []).append(e)

    exemplos = []
    for pagina_dados in extracao["pages"]:
        tokens_dict = pagina_dados["tokens"]
        if not tokens_dict:
            continue
        tokens = [t["text"] for t in tokens_dict]
        tokens_bbox = [t["bbox"] for t in tokens_dict]
        tags = ["O"] * len(tokens)
        for entidade in entidades_por_pagina.get(pagina_dados["page"], []):
            indices = indices_tokens_da_entidade(tokens_dict, entidade)
            primeiro = True
            for i in indices:
                if tags[i] != "O":
                    continue  # já rotulado por outra entidade (raro); evita conflito
                tags[i] = f"{'B' if primeiro else 'I'}-{entidade['type']}"
                primeiro = False

        # Divide a página em janelas deslizantes quando ultrapassa o orçamento de subtokens
        # do BERT -- sem isso, o trecho depois do corte (feito silenciosamente por
        # tokenizar_e_alinhar via MAX_LENGTH) nunca gera gradiente no treino, mesmo já
        # rotulado aqui em `tags` (achado 2026-09-15: página real média tem ~340 palavras/
        # ~970 subtokens, quase o dobro do limite -- ~28% das entidades do gabarito real
        # caíam nesse ponto cego). Cada janela vira um exemplo de treino independente; quando
        # a página cabe inteira no orçamento, gera um único exemplo -- comportamento idêntico
        # ao anterior a essa correção.
        comprimentos = comprimentos_subtokens(tokenizer, tokens)
        orcamento = MAX_LENGTH - 2  # reserva para [CLS]/[SEP]
        if sum(comprimentos) <= orcamento:
            janelas = [(0, len(tokens))]
        else:
            janelas = dividir_em_janelas_por_orcamento(comprimentos, orcamento, SOBREPOSICAO_PALAVRAS_JANELA)

        for idx_janela, (ini, fim) in enumerate(janelas):
            exemplos.append({
                "document_id": extracao["document_id"],
                "page": pagina_dados["page"],
                "janela": idx_janela,  # 0 quando a página cabe inteira numa janela só
                "tokens": tokens[ini:fim],
                "tags": tags[ini:fim],
                "tokens_bbox": tokens_bbox[ini:fim],  # mantido para a avaliação depois
            })
    return exemplos


# Junta o gabarito dos documentos reais (Gemini + alinhamento, notebook 03) com o dos
# sintéticos (gerado direto, notebook 00) — a não ser que INCLUIR_SINTETICOS=False (ver
# config), usado pra medir o efeito real do sintético comparando os dois treinos.
arquivos_gabarito = sorted(f for f in PASTA_GABARITO_REAIS.glob("*.json") if f.name != "_nao_alinhadas.json")
if INCLUIR_SINTETICOS:
    arquivos_gabarito += sorted(f for f in PASTA_GABARITO_SINTETICOS.glob("*.json") if f.name != "_nao_alinhadas.json")

if not arquivos_gabarito:
    raise SystemExit(f"Nenhum gabarito encontrado em {PASTA_GABARITO_REAIS} nem {PASTA_GABARITO_SINTETICOS}")

n_gabarito_reais = len(sorted(PASTA_GABARITO_REAIS.glob("*.json")))
n_gabarito_sinteticos = len(sorted(PASTA_GABARITO_SINTETICOS.glob("*.json"))) if INCLUIR_SINTETICOS else 0
print(f"Gabaritos encontrados: {n_gabarito_reais} reais + {n_gabarito_sinteticos} sintéticos "
      f"(INCLUIR_SINTETICOS={INCLUIR_SINTETICOS}) = {len(arquivos_gabarito)} total")

# Treina apenas com documentos cujo % de entidades alinhadas (notebook 03) seja >= LIMIAR_ALINHAMENTO_MINIMO_PCT —
# gabaritos com alinhamento muito ruim introduziriam bbox faltando/errada demais direto no treino.
# Documentos sintéticos sempre passam (100% match_type='exact' por construção).
todos_exemplos = []
docs_descartados_alinhamento = []
for arq_gab in arquivos_gabarito:
    arq_ext = PASTA_EXTRACAO / arq_gab.name
    if not arq_ext.exists():
        print(f"[AVISO] Sem extração correspondente para {arq_gab.name}, pulando.")
        continue
    gabarito = json.loads(arq_gab.read_text(encoding="utf-8"))
    entidades = gabarito.get("entities", [])
    if not entidades:
        continue
    # Duplicata de span físico (notebook 03, deduplicar_span_fisico) não é erro de alinhamento --
    # é uma segunda anotação do Gemini para a MESMA ocorrência física (ex. watermark/carimbo
    # sobreposto ao texto), descartada de propósito. Contá-la como "unmatched" aqui penalizaria
    # a decisão de entrada no treino por um motivo que não é falha de extração/alinhamento (ver
    # anotacoes.txt 2026-09-14) -- por isso ela é excluída do denominador, não tratada como falha.
    n_duplicatas = sum(
        1 for e in entidades
        if e.get("match_type") == "unmatched" and str(e.get("reason", "")).startswith("duplicata_span_fisico")
    )
    n_unmatched_genuino = sum(
        1 for e in entidades
        if e.get("match_type") == "unmatched" and not str(e.get("reason", "")).startswith("duplicata_span_fisico")
    )
    n_considerado = len(entidades) - n_duplicatas
    pct_alinhado = 100 * (n_considerado - n_unmatched_genuino) / n_considerado if n_considerado else 100.0
    if pct_alinhado < LIMIAR_ALINHAMENTO_MINIMO_PCT:
        docs_descartados_alinhamento.append((arq_gab.name, pct_alinhado, len(entidades)))
        continue
    extracao = json.loads(arq_ext.read_text(encoding="utf-8"))
    todos_exemplos.extend(construir_exemplos_documento(extracao, gabarito))

print(f"Documentos com >= {LIMIAR_ALINHAMENTO_MINIMO_PCT}% de alinhamento (usados no treino): "
      f"{len(arquivos_gabarito) - len(docs_descartados_alinhamento)}/{len(arquivos_gabarito)}")
if docs_descartados_alinhamento:
    print(f"Documentos descartados por alinhamento abaixo do limiar: {len(docs_descartados_alinhamento)}")
    for nome, pct, n_tot in sorted(docs_descartados_alinhamento, key=lambda x: x[1])[:10]:
        print(f"  - {nome}: {pct:.1f}% alinhado ({n_tot} entidades)")
    if len(docs_descartados_alinhamento) > 10:
        print(f"  ... e mais {len(docs_descartados_alinhamento) - 10}")

n_paginas_unicas = len({(ex["document_id"], ex["page"]) for ex in todos_exemplos})
n_paginas_com_janela_extra = len({
    (ex["document_id"], ex["page"]) for ex in todos_exemplos if ex["janela"] > 0
})
print(f"\nTotal de exemplos (janelas): {len(todos_exemplos)} -- de {n_paginas_unicas} paginas unicas")
if n_paginas_com_janela_extra:
    print(f"Paginas que passaram do orcamento de {MAX_LENGTH} subtokens e viraram multiplas "
          f"janelas: {n_paginas_com_janela_extra} (achado 2026-09-15, ver anotacoes.txt)")
n_com_entidade = sum(1 for ex in todos_exemplos if any(t != "O" for t in ex["tags"]))
print(f"Exemplos com pelo menos 1 entidade: {n_com_entidade}")

Gabaritos encontrados: 1000 reais + 0 sintéticos (INCLUIR_SINTETICOS=False) = 999 total
Documentos com >= 80% de alinhamento (usados no treino): 921/999
Documentos descartados por alinhamento abaixo do limiar: 78
  - cm2wc7vnq005010qd5ss082j5-Requerimento empresário.pdf.json: 0.0% alinhado (6 entidades)
  - cm3wbo2eb0002k2qd0ijf959z-PHOTO-2024-11-24-19-27-06 3.jpg.json: 0.0% alinhado (2 entidades)
  - cmbgj5p8m000v01qp52bu8zw9-ultimaalteracaopdf.json: 0.0% alinhado (25 entidades)
  - cmbv84d74007j01loazwh5p9s-contratosocialdeorigemgranpeaspdf.json: 0.0% alinhado (23 entidades)
  - cmiq5dmrg004j01nq3eqwhaoz-8alteraoconsolidadapdf.json: 0.0% alinhado (32 entidades)
  - cmiq5eo4w005201qu4y0h9a8c-altconsolidada031011pdf.json: 0.0% alinhado (32 entidades)
  - cmo1meazh003601s05o6tbmub-alteraocontratualbfccapitalpdf.json: 1.0% alinhado (313 entidades)
  - cmh9h3cun004g01lx9y6mb4u9-contratosocialpdf.json: 1.1% alinhado (92 entidades)
  - cm250epjp00537zqdhz597zq2-9ª Alteração Consolidada 2016

## Exemplo concreto: o que realmente entra no modelo

Todo o resto do gabarito (`occurrence_index`, `context_before`/`context_after`, `confidence`, `match_type`, `similarity_score`, `char_start`/`char_end`, `usage`) fica pra trás na etapa anterior -- só chega até aqui a palavra e a tag BIO de cada token da página. Dois níveis:
1. **Palavra + tag** -- o que `construir_exemplos_documento` monta (célula acima).
2. **Subpalavra + tag** -- depois do tokenizer do BERT quebrar cada palavra em subpalavras (wordpieces); é isso que vira `input_ids`/`labels` de verdade.

In [5]:
exemplo = next(ex for ex in todos_exemplos if any(t != "O" for t in ex["tags"]))
print(f"Documento: {exemplo['document_id']} -- página {exemplo['page']}\n")

print("1) PALAVRA -> TAG (só as com entidade, por brevidade):")
for palavra, tag in zip(exemplo["tokens"], exemplo["tags"]):
    if tag != "O":
        print(f"   {palavra:25s} {tag}")

print("\n2) SUBPALAVRA -> TAG, depois do tokenizer do BERT:")
encoding = tokenizer(exemplo["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
word_ids = encoding.word_ids()
subpalavras = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
anterior = None
for subpalavra, word_idx in zip(subpalavras, word_ids):
    if word_idx is None or exemplo["tags"][word_idx] == "O":
        anterior = word_idx
        continue
    tag = exemplo["tags"][word_idx]
    if word_idx == anterior:
        tag = "I-" + tag.split("-", 1)[1]  # continuação de subpalavra vira I-, mesmo se a palavra inteira é B-
    print(f"   {subpalavra:20s} {tag}")
    anterior = word_idx

print("\nÉ literalmente só isso (mais input_ids/attention_mask numéricos) que o BERT recebe --")
print("nenhum campo do gabarito bruto do Gemini chega até aqui.")


Documento: clpif4a2h00b92pqd67v27lm9-ALTERAÇÃO 1º DA EMPRESA TL TRANSPOSTES E LOGISTICA LTDA ME.pdf -- página 1

1) PALAVRA -> TAG (só as com entidade, por brevidade):
   IZALDO                    B-NOME
   JOS                       I-NOME
   É                         I-NOME
   DE                        I-NOME
   LIMA                      I-NOME
   896.928.374-91            B-CPF
   3.997.280                 B-RG
   rcio                      B-ENDERECO
   Chaves                    I-ENDERECO
   do                        I-ENDERECO
   R                         I-ENDERECO
   ê                         I-ENDERECO
   go                        I-ENDERECO
   ,                         I-ENDERECO
   n                         I-ENDERECO
   . º                       I-ENDERECO
   219                       I-ENDERECO
   ,                         I-ENDERECO
   Nova                      I-ENDERECO
   Escada                    I-ENDERECO
   ,                         I-ENDERECO
   Escada              

## Split treino/validação por documento

In [ ]:
doc_ids = sorted({ex["document_id"] for ex in todos_exemplos})
rng = random.Random(SEED)
rng.shuffle(doc_ids)

n_treino = round(len(doc_ids) * FRACAO_TREINO)
n_treino = min(max(n_treino, 1), len(doc_ids) - 1)  # garante pelo menos 1 doc em cada lado

docs_treino = set(doc_ids[:n_treino])
docs_validacao = set(doc_ids[n_treino:])

exemplos_treino = [ex for ex in todos_exemplos if ex["document_id"] in docs_treino]
exemplos_validacao = [ex for ex in todos_exemplos if ex["document_id"] in docs_validacao]

print(f"Split por documento: {FRACAO_TREINO:.0%} treino / {1 - FRACAO_TREINO:.0%} teste")
print(f"Documentos treino ({len(docs_treino)})")
print(f"Documentos teste ({len(docs_validacao)}): {sorted(docs_validacao)}")
print(f"Páginas treino: {len(exemplos_treino)} | Páginas teste: {len(exemplos_validacao)}")

## Tokenização e alinhamento de labels em subpalavras

In [ ]:
def tokenizar_e_alinhar(tokens: list, tags: list) -> dict:
    encoding = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    word_ids = encoding.word_ids()
    labels = []
    anterior = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != anterior:
            labels.append(LABEL2ID[tags[word_idx]])
        else:
            tag = tags[word_idx]
            if tag.startswith("B-"):
                tag = "I-" + tag[2:]
            labels.append(LABEL2ID[tag])
        anterior = word_idx
    encoding["labels"] = labels
    return encoding


class DatasetNER(torch.utils.data.Dataset):
    def __init__(self, exemplos):
        self.exemplos = exemplos

    def __len__(self):
        return len(self.exemplos)

    def __getitem__(self, idx):
        ex = self.exemplos[idx]
        return tokenizar_e_alinhar(ex["tokens"], ex["tags"])


dataset_treino = DatasetNER(exemplos_treino)
dataset_validacao = DatasetNER(exemplos_validacao)
data_collator = DataCollatorForTokenClassification(tokenizer)

## Busca de hiperparâmetros (10 cenários, todos com early stopping)

Cada cenário treina um BERTimbau do zero (checkpoint base `neuralmind/bert-base-portuguese-cased`) com sua própria combinação de `learning_rate`, `batch_size`, `weight_decay` e `warmup_ratio`. Todos usam o mesmo teto de `MAX_EPOCAS_TETO` épocas — o `EarlyStoppingCallback` decide quando parar de fato, monitorando o F1 de validação (paciência de `PACIENCIA_EARLY_STOPPING` avaliações sem melhora). Cada cenário é salvo em `models/bertimbau-pii-v1-cenarios/<nome>/`, junto com seus hiperparâmetros (`hiperparametros.json`).

In [ ]:
import time

from transformers import EarlyStoppingCallback, TrainerCallback


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [ID2LABEL[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    true_labels = [
        [ID2LABEL[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


class ProgressoEpocaCallback(TrainerCallback):
    """Imprime início/fim de cada época (com duração) e o resultado da avaliação,
    para acompanhar o treino em tempo real além da barra de progresso padrão."""

    def on_epoch_begin(self, args, state, control, **kwargs):
        self._inicio_epoca = time.time()
        epoca_atual = int(state.epoch) + 1 if state.epoch is not None else 1
        print(f"\n=== Época {epoca_atual}/{int(args.num_train_epochs)} iniciando... ===")

    def on_epoch_end(self, args, state, control, **kwargs):
        duracao = time.time() - self._inicio_epoca
        print(f"=== Época {state.epoch:.0f}/{int(args.num_train_epochs)} concluída em {duracao:.1f}s ===")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        partes = " | ".join(
            f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
            for k, v in logs.items()
            if k != "epoch"
        )
        print(f"  [passo {state.global_step}] {partes}")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics:
            return
        print(
            f"  >> avaliação: precision={metrics.get('eval_precision', 0):.3f} "
            f"recall={metrics.get('eval_recall', 0):.3f} f1={metrics.get('eval_f1', 0):.3f} "
            f"loss={metrics.get('eval_loss', 0):.4f}"
        )


# 10 cenários de hiperparâmetros — variando learning rate, batch size, weight decay e warmup.
# Todos com o mesmo teto de épocas; o early stopping decide quando parar de fato.
# OBS: esta versão do transformers (5.x) unificou warmup_ratio dentro de warmup_steps
# (aceita int = passos absolutos, ou float < 1 = fração dos passos totais).
MAX_EPOCAS_TETO = 30
PACIENCIA_EARLY_STOPPING = 3  # nº de avaliações (épocas) sem melhora no F1 antes de parar

CENARIOS = [
    {"nome": "c01_baseline",       "learning_rate": 3e-5, "batch_size": 4,  "weight_decay": 0.01, "warmup_ratio": 0.0},
    {"nome": "c02_lr_baixo",       "learning_rate": 2e-5, "batch_size": 4,  "weight_decay": 0.01, "warmup_ratio": 0.0},
    {"nome": "c03_lr_alto",        "learning_rate": 5e-5, "batch_size": 4,  "weight_decay": 0.01, "warmup_ratio": 0.0},
    {"nome": "c04_batch8",         "learning_rate": 3e-5, "batch_size": 8,  "weight_decay": 0.01, "warmup_ratio": 0.0},
    {"nome": "c05_batch16",        "learning_rate": 3e-5, "batch_size": 16, "weight_decay": 0.01, "warmup_ratio": 0.0},
    {"nome": "c06_sem_weight_decay", "learning_rate": 3e-5, "batch_size": 4, "weight_decay": 0.0, "warmup_ratio": 0.0},
    {"nome": "c07_weight_decay_alto", "learning_rate": 3e-5, "batch_size": 4, "weight_decay": 0.1, "warmup_ratio": 0.0},
    {"nome": "c08_warmup",         "learning_rate": 3e-5, "batch_size": 4,  "weight_decay": 0.01, "warmup_ratio": 0.1},
    {"nome": "c09_lrbaixo_batch8_warmup", "learning_rate": 1e-5, "batch_size": 8, "weight_decay": 0.01, "warmup_ratio": 0.1},
    {"nome": "c10_lralto_batch16_wdalto_warmup", "learning_rate": 5e-5, "batch_size": 16, "weight_decay": 0.1, "warmup_ratio": 0.1},
]

# Nome da pasta depende de INCLUIR_SINTETICOS -- evita sobrescrever os resultados do outro
# experimento (ver anotacoes.txt 2026-09-11) quando comparar com/sem sintéticos.
_sufixo_experimento = "-com-sinteticos" if INCLUIR_SINTETICOS else "-sem-sinteticos"
PASTA_CENARIOS = PASTA_MODELO_SAIDA.parent / f"bertimbau-pii-v1-cenarios{_sufixo_experimento}"
PASTA_CENARIOS.mkdir(parents=True, exist_ok=True)
print(f"Cenários serão salvos em: {PASTA_CENARIOS}")

resultados_cenarios = []

for cenario in CENARIOS:
    nome = cenario["nome"]
    pasta_cenario = PASTA_CENARIOS / nome
    pasta_cenario.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 70}\nCenário: {nome} -> {cenario}\n{'=' * 70}")

    modelo_cenario = AutoModelForTokenClassification.from_pretrained(
        CHECKPOINT_BASE,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    training_args = TrainingArguments(
        output_dir=str(pasta_cenario / "checkpoints"),
        num_train_epochs=MAX_EPOCAS_TETO,
        per_device_train_batch_size=cenario["batch_size"],
        per_device_eval_batch_size=cenario["batch_size"],
        learning_rate=cenario["learning_rate"],
        weight_decay=cenario["weight_decay"],
        warmup_steps=cenario["warmup_ratio"],  # float < 1 == fração dos passos totais (equivalente ao antigo warmup_ratio)
        eval_strategy="epoch",        # transformers 5.x renomeou evaluation_strategy -> eval_strategy
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_strategy="steps",
        logging_steps=50,
        disable_tqdm=False,
        seed=SEED,
        report_to=[],
    )

    trainer = Trainer(
        model=modelo_cenario,
        args=training_args,
        train_dataset=dataset_treino,
        eval_dataset=dataset_validacao,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[
            ProgressoEpocaCallback(),
            EarlyStoppingCallback(early_stopping_patience=PACIENCIA_EARLY_STOPPING, early_stopping_threshold=0.0),
        ],
    )

    inicio = time.time()
    trainer.train()
    duracao_treino = time.time() - inicio

    metricas_finais = trainer.evaluate()  # com load_best_model_at_end, avalia o melhor checkpoint do cenário

    trainer.save_model(str(pasta_cenario))
    tokenizer.save_pretrained(str(pasta_cenario))
    (pasta_cenario / "label_list.json").write_text(json.dumps(LABEL_LIST, ensure_ascii=False, indent=2), encoding="utf-8")
    (pasta_cenario / "hiperparametros.json").write_text(json.dumps(cenario, ensure_ascii=False, indent=2), encoding="utf-8")

    epocas_efetivas = round(trainer.state.epoch, 1) if trainer.state.epoch is not None else None

    resultado = {
        "nome": nome,
        **{k: v for k, v in cenario.items() if k != "nome"},
        "epocas_efetivas": epocas_efetivas,
        "duracao_treino_s": round(duracao_treino, 1),
        "eval_f1": metricas_finais.get("eval_f1"),
        "eval_precision": metricas_finais.get("eval_precision"),
        "eval_recall": metricas_finais.get("eval_recall"),
        "eval_loss": metricas_finais.get("eval_loss"),
        "pasta_modelo": str(pasta_cenario),
    }
    resultados_cenarios.append(resultado)

    print(
        f"\n>> {nome} concluído em {duracao_treino:.1f}s ({epocas_efetivas} épocas efetivas) "
        f"-> F1={resultado['eval_f1']:.4f} precision={resultado['eval_precision']:.4f} "
        f"recall={resultado['eval_recall']:.4f}"
    )

    # libera a GPU antes do próximo cenário
    del trainer, modelo_cenario
    torch.cuda.empty_cache()

print(f"\nTodos os {len(CENARIOS)} cenários concluídos.")

## Comparação entre os cenários

In [ ]:
resultados_ordenados = sorted(resultados_cenarios, key=lambda r: r["eval_f1"], reverse=True)

arq_comparacao = PASTA_CENARIOS / "_comparacao_cenarios.json"
arq_comparacao.write_text(json.dumps(resultados_ordenados, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Comparação salva em {arq_comparacao}")

cabecalho = (
    f"{'Cenário':<32}{'LR':<9}{'Batch':<7}{'WD':<6}{'Warmup':<8}"
    f"{'Épocas':<8}{'F1':<8}{'Precision':<11}{'Recall':<9}{'Tempo(s)':<9}"
)
print(f"\n{cabecalho}")
print("-" * len(cabecalho))
for r in resultados_ordenados:
    print(
        f"{r['nome']:<32}{r['learning_rate']:<9}{r['batch_size']:<7}{r['weight_decay']:<6}"
        f"{r['warmup_ratio']:<8}{r['epocas_efetivas']:<8}{r['eval_f1']:<8.4f}"
        f"{r['eval_precision']:<11.4f}{r['eval_recall']:<9.4f}{r['duracao_treino_s']:<9.1f}"
    )

melhor_cenario = resultados_ordenados[0]
print(f"\nMelhor cenário: {melhor_cenario['nome']} (F1={melhor_cenario['eval_f1']:.4f}) -> {melhor_cenario['pasta_modelo']}")

In [ ]:
import matplotlib.pyplot as plt

nomes = [r["nome"] for r in resultados_ordenados]
f1s = [r["eval_f1"] for r in resultados_ordenados]
precisions = [r["eval_precision"] for r in resultados_ordenados]
recalls = [r["eval_recall"] for r in resultados_ordenados]

x = list(range(len(nomes)))
largura = 0.27

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar([i - largura for i in x], precisions, width=largura, label="Precision")
ax.bar(x, recalls, width=largura, label="Recall")
ax.bar([i + largura for i in x], f1s, width=largura, label="F1")

ax.set_xticks(x)
ax.set_xticklabels(nomes, rotation=45, ha="right")
ax.set_ylabel("Score (nível de token, seqeval)")
ax.set_title("Comparação de hiperparâmetros — 10 cenários (early stopping), ordenado por F1")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

## Avaliação por bbox do melhor cenário (comparável com os notebooks 04/05)

As métricas da seção anterior são a nível de token (padrão seqeval), comparando os 10 cenários entre si. Para comparar de forma direta com os outros motores (spaCy, GLiNER), refazemos a avaliação por sobreposição de bbox (IoU) sobre as páginas de teste — reconstruindo entidades a partir das tags BIO previstas — usando apenas o **modelo do melhor cenário** (`melhor_cenario`).

In [ ]:
def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    caixas = []
    for g in grupos:
        caixas.append([min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)])
    return caixas


def decodificar_bio(tags_previstas: list, tokens_bbox: list) -> list:
    """Agrupa tags B-/I- consecutivas do mesmo tipo em entidades, convertendo para bbox."""
    entidades = []
    i = 0
    n = len(tags_previstas)
    while i < n:
        tag = tags_previstas[i]
        if tag == "O":
            i += 1
            continue
        tipo = tag.split("-", 1)[1]
        indices = [i]
        j = i + 1
        while j < n and tags_previstas[j] == f"I-{tipo}":
            indices.append(j)
            j += 1
        entidades.append({"type": tipo, "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices)})
        i = j
    return entidades


@torch.no_grad()
def prever_pagina(modelo, tokens: list) -> list:
    encoding = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(modelo.device) for k, v in encoding.items()}
    logits = modelo(**encoding).logits[0]
    pred_ids = logits.argmax(dim=-1).tolist()

    word_ids = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    tags_por_palavra = ["O"] * len(tokens)
    visto = set()
    for pid, word_idx in zip(pred_ids, word_ids):
        if word_idx is None or word_idx in visto:
            continue
        visto.add(word_idx)
        tags_por_palavra[word_idx] = ID2LABEL[pid]
    return tags_por_palavra


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(melhor_cenario["pasta_modelo"])
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print(f"Modelo carregado para avaliação por bbox: {melhor_cenario['nome']} ({melhor_cenario['pasta_modelo']})")

gabarito_por_doc_pagina = {}
for arq_gab in arquivos_gabarito:
    gabarito = json.loads(arq_gab.read_text(encoding="utf-8"))
    for e in gabarito["entities"]:
        if e.get("match_type") == "unmatched":
            continue
        gabarito_por_doc_pagina.setdefault((gabarito["document_id"], e["page"]), []).append(e)


def avaliar_paginas(exemplos: list) -> dict:
    """Roda a avaliação por bbox (IoU) sobre um subconjunto de páginas de teste e devolve os
    contadores tp/fp/fn por tipo — função extraída pra poder rodar separado em reais/sintéticos/geral."""
    contadores = {t: {"tp": 0, "fp": 0, "fn": 0} for t in TIPOS}
    for ex in exemplos:
        tags_previstas = prever_pagina(model, ex["tokens"])
        previstas = decodificar_bio(tags_previstas, ex["tokens_bbox"])
        gold = gabarito_por_doc_pagina.get((ex["document_id"], ex["page"]), [])

        previstas_por_tipo = {}
        for p in previstas:
            previstas_por_tipo.setdefault(p["type"], []).append(p)

        usadas = set()
        for g in gold:
            tipo = g["type"]
            candidatos = previstas_por_tipo.get(tipo, [])
            bbox_gold = bbox_externo(g.get("bbox") or [])
            melhor_iou, melhor_idx = 0.0, None
            for i, p in enumerate(candidatos):
                if id(p) in usadas:
                    continue
                score = iou(bbox_gold, bbox_externo(p["bbox"]))
                if score > melhor_iou:
                    melhor_iou, melhor_idx = score, i
            if melhor_idx is not None and melhor_iou >= IOU_MINIMO_MATCH:
                usadas.add(id(candidatos[melhor_idx]))
                contadores[tipo]["tp"] += 1
            else:
                contadores[tipo]["fn"] += 1

        for tipo, candidatos in previstas_por_tipo.items():
            for p in candidatos:
                if id(p) not in usadas:
                    contadores.setdefault(tipo, {"tp": 0, "fp": 0, "fn": 0})
                    contadores[tipo]["fp"] += 1
    return contadores


def imprimir_tabela(titulo: str, contadores: dict, n_docs: int, n_paginas: int):
    print(f"\n=== {titulo} ({n_docs} documento(s), {n_paginas} página(s)) ===")
    print(f"{'Tipo':<10}{'TP':<6}{'FP':<6}{'FN':<6}{'Precision':<12}{'Recall':<10}{'F1':<8}")
    print("-" * 62)
    tp_g = fp_g = fn_g = 0
    for tipo in TIPOS:
        c = contadores[tipo]
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        tp_g += tp; fp_g += fp; fn_g += fn
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        print(f"{tipo:<10}{tp:<6}{fp:<6}{fn:<6}{precision:<12.3f}{recall:<10.3f}{f1:<8.3f}")
    precision_g = tp_g / (tp_g + fp_g) if (tp_g + fp_g) else 0.0
    recall_g = tp_g / (tp_g + fn_g) if (tp_g + fn_g) else 0.0
    f1_g = 2 * precision_g * recall_g / (precision_g + recall_g) if (precision_g + recall_g) else 0.0
    print("-" * 62)
    print(f"{'GERAL':<10}{tp_g:<6}{fp_g:<6}{fn_g:<6}{precision_g:<12.3f}{recall_g:<10.3f}{f1_g:<8.3f}")


# Avaliação GERAL (todas as páginas de teste, reais + sintéticas misturadas — como já era feito)
contadores_geral = avaliar_paginas(exemplos_validacao)
imprimir_tabela("GERAL (reais + sintéticos)", contadores_geral, len(docs_validacao), len(exemplos_validacao))

# Quebra por ORIGEM — valida se o modelo generaliza pra reais de verdade, ou só decorou os
# templates sintéticos (ver anotacoes.txt, ressalva pendente da rodada anterior).
exemplos_validacao_reais = [ex for ex in exemplos_validacao if not ex["document_id"].startswith("sintetico_")]
exemplos_validacao_sinteticos = [ex for ex in exemplos_validacao if ex["document_id"].startswith("sintetico_")]
docs_validacao_reais = {d for d in docs_validacao if not d.startswith("sintetico_")}
docs_validacao_sinteticos = {d for d in docs_validacao if d.startswith("sintetico_")}

if exemplos_validacao_reais:
    contadores_reais = avaliar_paginas(exemplos_validacao_reais)
    imprimir_tabela("SÓ DOCUMENTOS REAIS de teste", contadores_reais, len(docs_validacao_reais), len(exemplos_validacao_reais))
else:
    print("\n[AVISO] Nenhum documento real no conjunto de teste -- não foi possível quebrar por origem.")

if exemplos_validacao_sinteticos:
    contadores_sinteticos = avaliar_paginas(exemplos_validacao_sinteticos)
    imprimir_tabela("SÓ DOCUMENTOS SINTÉTICOS de teste", contadores_sinteticos, len(docs_validacao_sinteticos), len(exemplos_validacao_sinteticos))
else:
    print("\n[AVISO] Nenhum documento sintético no conjunto de teste -- não foi possível quebrar por origem.")

print(
    "\nCompare o F1 GERAL com o F1 SÓ REAIS acima: se o F1 só-reais for bem menor que o geral, "
    "o ganho do treino com sintéticos está inflado por memorização dos próprios templates "
    "sintéticos, não é generalização real para documentos desconhecidos."
)